# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to C:\Users\Tetiana
[nltk_data]     Velehura\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Tetiana Velehura\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

d:\Projects\aie9\09_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
d:\Projects\aie9\09_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
d:\Projects\aie9\09_Synthetic_Data_Generation_and_LangSmith\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/14 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 8, relationships: 11)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 8, relationships: 11)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
Query synthesizers are responsible for combining retrieved information into a final answer. The three common types and their functions are:

The simple synthesizer takes the user's question and all the retrieved text chunks and sends them to the LLM in one single go to get an answer. It is fast and efficient but can fail if there is too much information to fit into the model's memory at once.

The refine synthesizer processes information step-by-step by looking at the first chunk of text to create an initial answer and then updating that answer as it reads through every following chunk. This is great for detail and accuracy because the model "re-thinks" the answer as it gets more data, though it takes more time and API calls.

The tree synthesizer organizes the retrieved chunks into a hierarchy or "tree" and summarizes pairs of chunks together into smaller versions until it reaches one final summary at the top. This is the best method for long-form summarization or when we need to combine themes from many different documents without losing the big picture.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"As a mental health educator, how does understa...",[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,Understanding the role of nutrition and diet i...,single_hop_specifc_query_synthesizer
1,What is appendix in mental health?,[13: The Science of Habit Formation Habits are...,The provided context does not include informat...,single_hop_specifc_query_synthesizer
2,How do shoulder shrugs serve as a practical ex...,[The Personal Wellness Guide A Comprehensive R...,Shoulder shrugs are recommended exercises to p...,single_hop_specifc_query_synthesizer
3,Considering the role of the United States in m...,[The Mental Health and Psychology Handbook A P...,The Mental Health and Psychology Handbook stat...,single_hop_specifc_query_synthesizer
4,What is the misspelled term related to therapy...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,"The term is 'DBT', which stands for Dialectica...",single_hop_specifc_query_synthesizer
5,How can adopting a growth mindset and practici...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,"Adopting a growth mindset, which involves beli...",multi_hop_abstract_query_synthesizer
6,How can adopting a growth mindset and practici...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,"Adopting a growth mindset, which emphasizes th...",multi_hop_abstract_query_synthesizer
7,"How does the mind-body connection, especially ...",[<1-hop>\n\nThe Mental Health and Psychology H...,The mind-body connection plays a significant r...,multi_hop_abstract_query_synthesizer
8,How can I use Chapter 4 and Chapter 20 to help...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,"Using Chapter 4, you can focus on building hea...",multi_hop_specific_query_synthesizer
9,How can understanding the science of habit for...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Understanding the science of habit formation f...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/18 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Whaat is the importance of EXERCISE AND MOVEME...,[The Personal Wellness Guide A Comprehensive R...,Exercise is one of the most important things y...,single_hop_specifc_query_synthesizer
1,What is the significance of sleep in maintaini...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"Sleep is crucial for physical health, mental w...",single_hop_specifc_query_synthesizer
2,What is Chapter 14 about,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 14: Morning Routines for Wellness disc...,single_hop_specifc_query_synthesizer
3,How does the mental health situation in the Un...,[The Mental Health and Psychology Handbook A P...,The context explains that mental health in the...,single_hop_specifc_query_synthesizer
4,How can practicing self-compassion when settin...,[<1-hop>\n\nsocial interactions How to set and...,Practicing self-compassion when setting bounda...,multi_hop_abstract_query_synthesizer
5,how stress management and relaxation technique...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,The context explains that building healthy hab...,multi_hop_abstract_query_synthesizer
6,How can social interactions and boundary-setti...,[<1-hop>\n\nsocial interactions How to set and...,Social interactions and boundary-setting are e...,multi_hop_abstract_query_synthesizer
7,How do the physical activity guidelines for ad...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,The physical activity guidelines for adults re...,multi_hop_abstract_query_synthesizer
8,Wha Chapter 1 and 21 do I need to do for health?,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,"Chapter 1 explains the basics of exercise, inc...",multi_hop_specific_query_synthesizer
9,How do Chapters 1 and 8 together emphasize the...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Chapter 1 highlights the significance of under...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
The unrolled approach gives full manual control over every step of the pipeline, allowing to hand-pick documents, define specific transformations, and fine-tune the distribution of question types. While this provides high precision and transparency, it is much more time-consuming and requires a deeper understanding of the underlying components.

On the other hand, the abstracted approach uses automated workflows to handle the heavy lifting, making it significantly faster to generate large datasets with minimal setup. The trade-off here is a loss of granular control and the risk of generating generic or repetitive queries if the automation isn't perfectly tuned to the specific domain.

We should choose the unrolled approach when we are working on a specialized production system where data quality is critical and we need to ensure specific edge cases are covered. We should opt for the abstracted approach during the early stages of prototyping.

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [41]:
### Define custom query distribution with different weights ###

# Create a custom distribution emphasizing multi-hop queries
# This is useful when you want to evaluate RAG systems on complex reasoning tasks
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),      # 20% single-hop
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),       # 40% multi-hop abstract
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),       # 40% multi-hop specific
]

print("Default distribution:")
print("  SingleHopSpecificQuerySynthesizer: 50%")
print("  MultiHopAbstractQuerySynthesizer: 25%")
print("  MultiHopSpecificQuerySynthesizer: 25%")
print("\nCustom distribution (emphasizing complex queries):")
print("  SingleHopSpecificQuerySynthesizer: 20%")
print("  MultiHopAbstractQuerySynthesizer: 40%")
print("  MultiHopSpecificQuerySynthesizer: 40%")

# Generate test set with custom distribution
print("\n" + "="*60)
print("Generating test set with CUSTOM distribution...")
print("="*60)
custom_testset = generator.generate(testset_size=10, query_distribution=custom_query_distribution)

# Generate test set with default distribution for comparison
print("\n" + "="*60)
print("Generating test set with DEFAULT distribution...")
print("="*60)
default_testset = generator.generate(testset_size=10, query_distribution=query_distribution)

# Convert to pandas for analysis
custom_df = custom_testset.to_pandas()
default_df = default_testset.to_pandas()

print("\n" + "="*60)
print("COMPARISON: DEFAULT vs CUSTOM DISTRIBUTION")
print("="*60)

# Display available columns
print("\nAvailable columns:", custom_df.columns.tolist())

print("\n📊 DEFAULT Distribution Test Set:")
print(default_df[['user_input', 'reference']].head(10).to_string())

print("\n\n📊 CUSTOM Distribution Test Set:")
print(custom_df[['user_input', 'reference']].head(10).to_string())

# Analyze question characteristics in each distribution
print("\n" + "="*60)
print("QUESTION ANALYSIS")
print("="*60)

print("\nDefault Distribution - Sample Questions:")
for idx, row in default_df.head(5).iterrows():
    print(f"  Q{idx+1}: {row['user_input'][:80]}...")

print("\n\nCustom Distribution - Sample Questions:")
for idx, row in custom_df.head(5).iterrows():
    print(f"  Q{idx+1}: {row['user_input'][:80]}...")

# Compare question complexity
print("\n" + "="*60)
print("QUESTION COMPLEXITY METRICS")
print("="*60)

# Calculate average question length as proxy for complexity
default_avg_len = default_df['user_input'].str.len().mean()
custom_avg_len = custom_df['user_input'].str.len().mean()

print(f"\nDefault Distribution - Avg Question Length: {default_avg_len:.1f} chars")
print(f"Custom Distribution - Avg Question Length: {custom_avg_len:.1f} chars")

# Count questions by length category
def categorize_complexity(text):
    length = len(text)
    if length < 50:
        return "Simple"
    elif length < 100:
        return "Moderate"
    else:
        return "Complex"

default_df['complexity'] = default_df['user_input'].apply(categorize_complexity)
custom_df['complexity'] = custom_df['user_input'].apply(categorize_complexity)

print("\nDefault Distribution - Question Complexity:")
print(default_df['complexity'].value_counts().to_string())

print("\n\nCustom Distribution - Question Complexity:")
print(custom_df['complexity'].value_counts().to_string())

# Summary explanation
print("\n" + "="*60)
print("EXPLANATION OF DESIGN CHOICES")
print("="*60)
print("""
Why I chose this custom distribution:

1. **Reduced Single-Hop (50% → 20%)**: Single-hop queries (simple lookups) are less 
   challenging and may not effectively test RAG retrieval quality. By reducing this, 
   we create a more demanding evaluation set that focuses on complex reasoning.

2. **Increased Multi-Hop Abstract (25% → 40%)**: Abstract multi-hop questions require 
   connecting multiple concepts and reasoning at a higher level. This better tests 
   the system's ability to synthesize information and understand relationships between
   different parts of the knowledge base.

3. **Increased Multi-Hop Specific (25% → 40%)**: Specific multi-hop queries combine 
   multiple documents to answer detailed questions. This is closer to real-world 
   complex reasoning tasks and provides better coverage of difficult scenarios.

Use Cases:
- **DEFAULT (50/25/25)**: Good for balanced evaluation of basic to intermediate tasks,
  general-purpose RAG system testing
- **CUSTOM (20/40/40)**: Better for evaluating advanced RAG systems, complex reasoning,
  systems designed for multi-document analysis, and production scenarios where query
  complexity matters
""")


Default distribution:
  SingleHopSpecificQuerySynthesizer: 50%
  MultiHopAbstractQuerySynthesizer: 25%
  MultiHopSpecificQuerySynthesizer: 25%

Custom distribution (emphasizing complex queries):
  SingleHopSpecificQuerySynthesizer: 20%
  MultiHopAbstractQuerySynthesizer: 40%
  MultiHopSpecificQuerySynthesizer: 40%

Generating test set with CUSTOM distribution...


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]


Generating test set with DEFAULT distribution...


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]


COMPARISON: DEFAULT vs CUSTOM DISTRIBUTION

Available columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

📊 DEFAULT Distribution Test Set:
                                                                                                                                                                                                                                                                                                                                       user_input                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'notable-flag-30' at:
https://smith.langchain.com/o/240f82da-89c1-4f91-8a48-75c8cc44cdf8/datasets/65fa50b9-86b2-41fe-80cb-fe96ee7a6115/compare?selectedSessions=029bf541-b959-499c-aafd-4b501d1ff90d




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How can combining Cognitive Behavioral Therapy...,Combining Cognitive Behavioral Therapy (CBT) a...,None,Integrating Cognitive Behavioral Therapy (CBT)...,True,True,True,3.945497,8f23cfc8-c5b4-4a4a-ad49-2b51c63d383b,019c7146-b333-78b1-b4cc-18e8c112ee75
1,how do chapter 9 and chapter 14 connect about ...,I don't know.,None,chapter 9 talks about sleep and how good sleep...,False,False,False,1.468271,28bf42c6-74a3-40e7-ad66-dadb60ca628e,019c7146-fee8-7713-9b7e-9b41c0a5ab58
2,How do Chapters 1 and 8 together emphasize the...,Chapters 1 and 8 together emphasize overall we...,None,Chapter 1 highlights the significance of under...,True,True,False,3.324146,4a85ad11-0e34-4975-a299-e7b9799f8e86,019c7147-2f56-7683-a3a0-0c78eda5528b
3,Wha Chapter 1 and 21 do I need to do for health?,Based on the provided context:\n\n- Chapter 1 ...,None,"Chapter 1 explains the basics of exercise, inc...",True,True,True,5.212026,78e4d28f-6542-4ef5-a0cc-6b0a82cdef6a,019c7147-7fe5-76d1-9694-55cfbb99c07f
4,How do the physical activity guidelines for ad...,The physical activity guidelines for adults re...,None,The physical activity guidelines for adults re...,True,True,False,7.131267,5a393557-333a-496c-b238-1258ad534d70,019c7147-ca42-7e72-ad45-f0505f58ca58
5,How can social interactions and boundary-setti...,Social interactions and boundary-setting can h...,None,Social interactions and boundary-setting are e...,True,True,True,4.701226,8b0edea1-8dba-4e7e-8d47-da9856545b1f,019c7148-1a03-7da2-9952-ce3932b13117
6,how stress management and relaxation technique...,"Based on the provided context, stress manageme...",None,The context explains that building healthy hab...,True,True,True,5.180743,9754967d-069d-42f8-b656-f87daf61b6e2,019c7148-620d-7dd0-a3e1-c1d179550258
7,How can practicing self-compassion when settin...,Practicing self-compassion when setting bounda...,None,Practicing self-compassion when setting bounda...,True,True,True,3.236523,dd5d053e-eb98-42c8-96ea-0c8b1f995860,019c7148-c38b-77d0-aa09-2a337e45d5c1
8,How does the mental health situation in the Un...,I don't know.,None,The context explains that mental health in the...,False,False,False,1.058901,c247d772-0873-4771-86b6-8448017f2c21,019c7148-ffab-7c32-a385-f6b233526b75
9,What is Chapter 14 about,I don't know.,None,Chapter 14: Morning Routines for Wellness disc...,False,False,False,1.412320,a32a0080-8089-48ea-878a-bf93bb9e6cdf,019c7149-2c7e-70f2-a76b-72ae5b7fda43


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
If the chunks are too small, the AI misses the full story because the beginning and end of a concept get cut off. If the chunks are too big, the AI gets overwhelmed by "filler" text and struggles to find the specific fact it actually needs. Changing the size changes how well the AI can "see" the information - we're trying to find the sweet spot where it has enough context to understand the topic but not so much that it gets distracted or confused.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Larger models generally have a more sophisticated understanding of language, meaning they can catch subtle relationships between words that a smaller model might miss. By using the larger version, the agent becomes much better at finding the most relevant information because its mathematical map of the data is more precise, though the trade-off is usually a slight increase in the time and compute power needed to process those searches.

In [35]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, time to level up your sleep game like a total sleep ninja! 🛌💥 Based on the ultimate wisdom in the sleep cosmos:\n\n1. **Lock in a consistent sleep schedule, no matter what day it is** — weekends don’t get to mess with your rhythm. Your body LOVES routine.\n\n2. **Craft a chill bedtime ritual** — think reading a dope book, some gentle stretching, or soaking in a warm bath to ease your mind into dreamland vibes.\n\n3. **Make your bedroom THE sleep fortress** — keep it cool (65-68°F / 18-20°C), dark (blackout curtains or a slick sleep mask), and quiet (white noise machine or earplugs if needed).\n\n4. **Dump screens at least 1-2 hours before bed** — blue light is the enemy of your melatonin game.\n\n5. **Drop caffeine by 2 PM** — caffeine late in the day will sabotage your snooze time hard.\n\n6. **Sweat smartly** — get your body moving regularly but steer clear of heavy workouts close to bedtime.\n\n7. **Watch your nighttime fuel** — minimize alcohol and heavy meals pre-sleep s

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'terrific-wound-90' at:
https://smith.langchain.com/o/240f82da-89c1-4f91-8a48-75c8cc44cdf8/datasets/65fa50b9-86b2-41fe-80cb-fe96ee7a6115/compare?selectedSessions=7dd1a78e-75d5-45b8-97cc-e90c1ec79a81




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How can combining Cognitive Behavioral Therapy...,"Alright, buckle up for a mind-melding combo of...",None,Integrating Cognitive Behavioral Therapy (CBT)...,True,True,True,9.234340,8f23cfc8-c5b4-4a4a-ad49-2b51c63d383b,019c714a-2065-79a3-a573-5cd58230fc81
1,how do chapter 9 and chapter 14 connect about ...,"Yo, here’s the slick connection between Chapte...",None,chapter 9 talks about sleep and how good sleep...,True,True,True,4.946477,28bf42c6-74a3-40e7-ad66-dadb60ca628e,019c714a-79b7-7220-886c-076a026ad34b
2,How do Chapters 1 and 8 together emphasize the...,"Alright, let’s crank this up to eleven with th...",None,Chapter 1 highlights the significance of under...,True,True,True,6.504162,4a85ad11-0e34-4975-a299-e7b9799f8e86,019c714a-c6a1-7ae3-9475-82ef9407fbb1
3,Wha Chapter 1 and 21 do I need to do for health?,"Yo, diving into Chapter 1 from the Personal We...",None,"Chapter 1 explains the basics of exercise, inc...",False,False,True,3.156483,78e4d28f-6542-4ef5-a0cc-6b0a82cdef6a,019c714b-20c5-7360-9713-506420cbdd1f
4,How do the physical activity guidelines for ad...,"Alright, let’s crank this wellness wattage up ...",None,The physical activity guidelines for adults re...,True,True,True,12.950974,5a393557-333a-496c-b238-1258ad534d70,019c714b-61ee-7491-9af5-4e82bc1f5837
5,How can social interactions and boundary-setti...,"Alright, buckle up because mental health maste...",None,Social interactions and boundary-setting are e...,True,True,True,6.696936,8b0edea1-8dba-4e7e-8d47-da9856545b1f,019c714b-c87e-7621-8d18-8a9d3c2784c2
6,how stress management and relaxation technique...,"Alright, buckle up for some next-level wellnes...",None,The context explains that building healthy hab...,True,True,True,6.753054,9754967d-069d-42f8-b656-f87daf61b6e2,019c714c-1d2e-7603-b4e4-db8dc07c0b06
7,How can practicing self-compassion when settin...,"Alright, here’s the high-vibe answer for you:\...",None,Practicing self-compassion when setting bounda...,True,True,True,5.101097,dd5d053e-eb98-42c8-96ea-0c8b1f995860,019c714c-766c-7ae2-a3da-786ad6400e89
8,How does the mental health situation in the Un...,"Yo, let’s break it down: In the U.S., anxiety ...",None,The context explains that mental health in the...,True,True,True,5.451366,c247d772-0873-4771-86b6-8448017f2c21,019c714c-b92e-7292-a462-47d6a9591db9
9,What is Chapter 14 about,Chapter 14 is all about owning your mornings l...,None,Chapter 14: Morning Routines for Wellness disc...,True,True,True,2.876343,a32a0080-8089-48ea-878a-bf93bb9e6cdf,019c714d-0872-7193-8911-f28b2acb42f9


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:


Based on the evaluation results from LangSmith, switching from the first chain (notable-flag-30) to the second chain (terrific-wound-90) significantly improved the agent's performance across several key metrics.

Performance Analysis
Dopeness (0.417 → 1.00): This metric saw the largest increase because the second chain adopted a high-energy, personality-driven "vibe" (using phrases like "Yo," "Alright, buckle up," and "high-vibe answer"). The first chain was too robotic and dry, resulting in many 0.00 scores for style.

Helpfulness (0.75 → 0.917): The second chain became much more effective at answering specific questions it previously failed. For example, in rows 1, 10, and 11, the first chain responded with "I don't know," whereas the second chain successfully retrieved and summarized the information regarding Chapter 14 and sleep connections.

QA Accuracy (0.75 → 0.917): The overall accuracy improved because the second chain's retrieval or reasoning logic was better tuned. It stopped hallucinating ignorance (saying "I don't know") and started providing factual content that matched the reference outputs, even if it did fail on one specific query regarding Chapter 21 (row 6).

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores